# Nomos Integrated Master System
**End-to-End Pipeline: Data Ingestion → Feature Engineering → Regime Detection**

This notebook serves as the unified research lab for Project Nomos. It handles the entire multi-asset data flow and trains the Hidden Markov Model (HMM) for regime detection.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Ensure the project root is in the path for src imports
sys.path.append(os.path.abspath('..'))

from src.data.data_manager import DataManager
from src.models.hmm_model import NomosHMM

# Styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['axes.titlesize'] = 16

## 1. Unified Data Pipeline (Phase 1)
This block fetches data from Yahoo Finance (NIFTY, Gold, USDINR) and Kite Connect (IndiaVIX), then applies log-transforms, detrending, and stationarity enforcement.

In [ ]:
manager = DataManager("../config/config.yaml")

print("[*] Step 1.1: Fetching Raw Market Data...")
df_raw = manager.fetch_all_data(start_date="2008-01-01")

print("[*] Step 1.2: Processing Features (Returns, Spreads, Detrending)...")
df_features = manager.process_data(df_raw)

print("[*] Step 1.3: Normalizing Features (Z-Score Scaling)...")
df_final = manager.scale_features(df_features)

print("\n--- Stationarity Diagnostics Report ---")
stat_report = manager.check_features(df_features)
display(stat_report)

print(f"\nPipeline complete. Feature Matrix Ready: {df_final.shape}")

## 2. Feature Visualization
Visual check of the engineered signals before feeding them into the HMM.

In [ ]:
visual_features = [c for c in df_features.columns if '_Ret' in c or 'Spread' in c or 'Detrended' in c]
df_features[visual_features].plot(subplots=True, figsize=(15, 12), title="Raw Engineered Features", cmap='viridis')
plt.tight_layout()
plt.show()

## 3. Regime Detection (Phase 2)
Training the Hidden Markov Model. We use defensive logic to skip missing features if one of the data sources failed.

In [ ]:
# Define intended features
requested = ['NIFTY50_Ret', 'VIX_Spread', 'Corr_NIFTY50_Gold', 'Detrended_Vol']
actual_features = [f for f in requested if f in df_final.columns]

print(f"[*] Training HMM on features: {actual_features}")
if 'VIX_Spread' not in actual_features:
    print("!!! WARNING: VIX_Spread missing. Please check Kite Connect access_token in config.yaml")

# Initialize and Fit
hmm_model = NomosHMM(n_components=3)
hmm_model.fit(df_final, actual_features)

print("\n[*] Model Converged!")
print("[*] Interpretation Mapping:", hmm_model.state_labels)

# Save model for strategy development
hmm_model.save_model("../models/nomos_hmm_latest.joblib")

## 4. Final Results: Regime Overlay
Mapping the detected states back to the Nifty 50 cumulative returns path.

In [ ]:
# Predict states
df_final['Regime'] = hmm_model.predict(df_final)

# Create the Master Regime Chart
colors = {'Bull': '#2ecc71', 'Neutral': '#95a5a6', 'Bear': '#e74c3c'}

plt.figure(figsize=(15, 9))
for label, color in colors.items():
    mask = df_final['Regime'] == label
    if mask.any():
        plt.scatter(df_final.index[mask], df_final.loc[mask, 'NIFTY50_Ret'].cumsum(), 
                    label=label, c=color, s=12, alpha=0.8)

plt.title("Nomos Regime Detection: NIFTY 50 (2008 - Present)", fontsize=18, fontweight='bold')
plt.ylabel("Cumulative Log-Returns", fontsize=14)
plt.legend(title="Market State", fontsize=12, loc='upper left')

# Highlight the Transition Matrix
plt.show()

print("\n--- Regime Persistent Matrix (Stability) ---")
labels = [hmm_model.state_labels[i] for i in range(3)]
trans_df = pd.DataFrame(hmm_model.model.transmat_, index=labels, columns=labels)
display(trans_df.style.background_gradient(cmap='Greens'))